In [16]:
from pathlib import Path
import ast

import pandas as pd
from datetime import datetime

ROOT = Path.cwd().parent
CLEAN_FILE = ROOT / "data" / "clean" / "movies_clean.csv"

df = pd.read_csv(CLEAN_FILE)

In [17]:
df["release_date"] = pd.to_datetime(df["release_date"],errors="coerce")

df["genres"] = df["genres"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

df["keywords"] = df["keywords"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

In [18]:
df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
df["release_decade"] = (df["release_year"] // 10) * 10

In [19]:
df[["release_date", "release_year", "release_month", "release_decade"]].head()

,release_date,release_year,release_month,release_decade
0,2026-07-29,2026.0,7.0,2020.0
1,2026-10-01,2026.0,10.0,2020.0
2,2026-09-16,2026.0,9.0,2020.0
3,2026-07-15,2026.0,7.0,2020.0
4,2026-09-10,2026.0,9.0,2020.0


In [20]:
df["roi"] = df["revenue"] / df["budget"]
df["movie_age"] = datetime.now().year - df["release_year"]

In [21]:
df["genre_count"] = df["genres"].apply(len)

In [22]:
df[["title", "genres", "genre_count"]].head()

,title,genres,genre_count
0,Spider-Man: Brand New Day,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",3
1,Renegade Immortal: Battle of the Immortal Slayer,"[{'id': 16, 'name': 'Animation'}, {'id': 28, '...",4
2,Resident Evil,"[{'id': 27, 'name': 'Horror'}, {'id': 878, 'na...",3
3,The Odyssey,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",3
4,The Fix,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...",2


In [23]:
df["keyword_count"] = df["keywords"].apply(len)

In [24]:
df[["title", "keywords", "keyword_count"]].head()

,title,keywords,keyword_count
0,Spider-Man: Brand New Day,"[{'id': 9678, 'name': 'mind control'}, {'id': ...",19
1,Renegade Immortal: Battle of the Immortal Slayer,"[{'id': 179411, 'name': 'based on fairy tale'}...",4
2,Resident Evil,"[{'id': 9744, 'name': 'outbreak'}, {'id': 1034...",16
3,The Odyssey,"[{'id': 3799, 'name': 'ship'}, {'id': 3265, 'n...",18
4,The Fix,"[{'id': 591, 'name': 'central intelligence age...",6


In [25]:
df["runtime_category"] = pd.cut(
    df["runtime"],
    bins=[0, 90, 120, 150, float("inf")],
    labels=["Short", "Medium", "Long", "Very Long"],
    include_lowest=True,
)

In [26]:
df[["title", "runtime", "runtime_category"]].head(10)

,title,runtime,runtime_category
0,Spider-Man: Brand New Day,145.0,Long
1,Renegade Immortal: Battle of the Immortal Slayer,NaN,NaN
2,Resident Evil,95.0,Medium
3,The Odyssey,173.0,Very Long
4,The Fix,142.0,Long
5,Runner,97.0,Medium
6,Coyote vs. Acme,103.0,Medium
7,After Impact,95.0,Medium
8,The End of Oak Street,100.0,Medium
9,Digger,129.0,Long


In [27]:
df["runtime_category"].value_counts()

runtime_category
Medium       1677
Long          716
Short         374
Very Long     163
Name: count, dtype: int64

In [28]:
df["overview_length"] = df["overview"].fillna("").str.len()

In [32]:
df[
    [
        "release_year",
        "release_month",
        "release_decade",
        "genre_count",
        "keyword_count",
        "runtime_category",
        "overview_length",
        "roi",
        "movie_age"
    ]
].head()

,release_year,release_month,release_decade,genre_count,keyword_count,runtime_category,overview_length,roi,movie_age
0,2026.0,7.0,2020.0,3,19,Long,370,11.135453,0.0
1,2026.0,10.0,2020.0,4,4,NaN,250,NaN,0.0
2,2026.0,9.0,2020.0,3,16,Medium,133,3.272000,0.0
3,2026.0,7.0,2020.0,3,18,Very Long,280,7.069856,0.0
4,2026.0,9.0,2020.0,2,6,Long,154,NaN,0.0


In [30]:
df[
    [
        "release_year",
        "release_month",
        "release_decade",
        "genre_count",
        "keyword_count",
        "runtime_category",
        "overview_length"
    ]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 2963 entries, 0 to 2962
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   release_year      2960 non-null   float64 
 1   release_month     2960 non-null   float64 
 2   release_decade    2960 non-null   float64 
 3   genre_count       2963 non-null   int64   
 4   keyword_count     2963 non-null   int64   
 5   runtime_category  2930 non-null   category
 6   overview_length   2963 non-null   int64   
dtypes: category(1), float64(3), int64(3)
memory usage: 142.1 KB


In [31]:
FEATURE_DIR = ROOT / "data" / "features"
FEATURE_FILE = FEATURE_DIR / "movies_features.csv"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(FEATURE_FILE, index=False)

print(f"Feature dataset saved to: {FEATURE_FILE}")

Feature dataset saved to: /home/elgmouri/projects/moviesHub/data/features/movies_features.csv
